In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import MultinomialNB

from sklearn.svm import LinearSVC

from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import StackingClassifier

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline

from sklearn.metrics import classification_report

In [2]:
#load data
df = pd.read_csv("/home/sahil-gaund/Desktop/scamshield-ai/datasets/processed/cleaned_spam.csv")
df.head()

,label,message,clean_message
0,spam,Congratulations! You've been selected for a lu...,congratul select luxuri vacat getaway claim prize
1,spam,URGENT: Your account has been compromised. Cli...,urgent account compromis click reset password ...
2,spam,You've won a free iPhone! Claim your prize by ...,free iphon claim prize click link
3,spam,Act now and receive a 50% discount on all purc...,act receiv discount purchas limit time offer
4,spam,Important notice: Your subscription will expir...,import notic subscript expir soon renew avoid ...


In [3]:
df['label'] = df['label'].map({'ham':0, 'spam':1})

In [4]:
df.head()

,label,message,clean_message
0,1,Congratulations! You've been selected for a lu...,congratul select luxuri vacat getaway claim prize
1,1,URGENT: Your account has been compromised. Cli...,urgent account compromis click reset password ...
2,1,You've won a free iPhone! Claim your prize by ...,free iphon claim prize click link
3,1,Act now and receive a 50% discount on all purc...,act receiv discount purchas limit time offer
4,1,Important notice: Your subscription will expir...,import notic subscript expir soon renew avoid ...


In [15]:
# REMOVE NULL VALUES
df.dropna(inplace=True)

# CONVERT TO STRING

df['clean_message'] = df['clean_message'].astype(str)

# FEATURES & TARGET

X = df['clean_message']

y = df['label']

In [16]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [17]:
#TF-IDf
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

In [18]:
#base model 
base_models = [
    ('lr', LogisticRegression()),
    
    ('nb', MultinomialNB()),
    
    ('svm', LinearSVC()),
    
    ('rf', RandomForestClassifier())
]

In [19]:
#meta model 
meta_model = XGBClassifier(eval_metric='logloss')

In [20]:
#stacking ensemble
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

In [21]:
#final pipline 
pipeline = Pipeline([
    ('tfidf', tfidf),
    ('model', stack_model)
])

In [22]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [23]:
#predticiions

y_pred = pipeline.predict(X_test)

In [24]:
#evaluation
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1710
           1       0.93      0.89      0.91       477

    accuracy                           0.96      2187
   macro avg       0.95      0.94      0.94      2187
weighted avg       0.96      0.96      0.96      2187



In [25]:
joblib.dump(
    pipeline,"/home/sahil-gaund/Desktop/scamshield-ai/ml/models/scamshield_model.pkl"
)
print("model saved")

model saved


In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred))

print("Recall:", recall_score(y_test, y_pred))

print("F1:", f1_score(y_test, y_pred))

print("ROC AUC:", roc_auc_score(y_test, y_pred))

Accuracy: 0.9615912208504801
Precision: 0.9281045751633987
Recall: 0.8930817610062893
F1: 0.9102564102564102
ROC AUC: 0.9368917576961272
